# Feature ideas

- `num_trials_conducted`: number of trials conducted at this site in the past
- `num_conditions_conducted`: number of different diseases conducted at this site in the past
- `num_interventions_conducted`: number of different drugs conducted at this site in the past
- `num_sponsors_conducted`: number of different sponsors conducted at this site in the past

- `num_phase1_trials_conducted`: number of trials conducted at this site in the past
- `num_phase2_trials_conducted`: number of trials conducted at this site in the past
- `num_phase3_trials_conducted`: number of trials conducted at this site in the past
- `num_phase4_trials_conducted`: number of trials conducted at this site in the past

- `avg_historical_trial_success_rate`: past trial history success rate
- `avg_historical_trial_adverse_events`: past trial adverse event amount

- `max_historical_trial_enrollment`: maximum number of past trial number of participants; ideally a trial site with capacity to do large scale trial should be more mature
- `avg_historical_trial_enrollment`: average number of past trial number of participants; ideally a trial site with capacity to do large scale trial should be more mature

- `operation_years`: number of opreration years defined as number of years from the first to last trial

- `in_us`: if the trial facility locates in US
- `in_china`: if the trial facility locates in china
- `in_germany`: if the trial facility locates in germany
- `in_france`: if the trial facility locates in germany
- `in_italy`: if the trial facility locates in germany
- `in_japan`: if the trial facility locates in germany
- `in_canada`: if the trial facility locates in germany
- `in_spain`: if the trial facility locates in germany
- `in_uk`: if the trial facility locates in germany
- `in_russia`: if the trial facility locates in germany

- `is_in_california`: if the trial facility locates in US california
- `is_in_florida`: if the trial facility locates in US florida
- `is_in_texas`: if the trial facility locates in US texas
- `is_in_ny`: if the trial facility locates in US ny
- `is_in_penn`: if the trial facility locates in US penn
- `is_in_nc`: if the trial facility locates in US north carolina

- `years_since_last_trial_facility`: how recent is the lastest trial

- `in_multi_center_trial`: has this site be part of a larger multi-center study?
- `num_multi_center_trial`: how many times have this trial be part of a larger multi-center study?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from relbench.datasets import get_dataset
from relbench.tasks import get_task

from tqdm import tqdm 
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True, nb_workers = 10)

%load_ext autoreload
%autoreload 2


In [ ]:
def get_features(train_df):
    dataset = get_dataset(name='rel-trial', process=False)
    task = get_task('rel-trial', 'site-success', download=True)
    tables = dataset.get_db().table_dict

    interventions_studies = tables['interventions_studies'].df
    conditions_studies = tables['conditions_studies'].df
    sponsors_studies = tables['sponsors_studies'].df
    facilities_studies = tables['facilities_studies'].df
    studies = tables['studies'].df
    designs = tables['designs'].df
    eligibilities = tables['eligibilities'].df
    sponsors = tables['sponsors'].df
    reported_event_totals = tables["reported_event_totals"].df
    
    outcomes = tables['outcomes'].df
    facilities = tables['facilities'].df
    outcome_analyses = tables['outcome_analyses'].df
    outcome_analyses = outcome_analyses.merge(outcomes[['id', 'outcome_type']], left_on='outcome_id', right_on = 'id')

    def get_derived_features(facility_id, timestamp):
        try:
            history_ncts = facilities_studies[(facilities_studies.facility_id == facility_id) & (facilities_studies.date < timestamp)].nct_id.unique()
            num_trials_conducted = len(history_ncts)
            num_conditions_conducted = len(conditions_studies[conditions_studies.nct_id.isin(history_ncts)].condition_id.unique())
            num_interventions_conducted = len(interventions_studies[interventions_studies.nct_id.isin(history_ncts)].intervention_id.unique())
            num_sponsors_conducted = len(sponsors_studies[sponsors_studies.nct_id.isin(history_ncts)].sponsor_id.unique())
        
            studies_temp = studies[studies.nct_id.isin(history_ncts)]
            studies_temp['phase'] = studies_temp.phase.astype(str)
            num_phase1 = len(studies_temp[studies_temp.phase.str.contains('1')])
            num_phase2 = len(studies_temp[studies_temp.phase.str.contains('2')])
            num_phase3 = len(studies_temp[studies_temp.phase.str.contains('3')])
            num_phase4 = len(studies_temp[studies_temp.phase.str.contains('4')])
        
            outcome_temp = outcome_analyses[outcome_analyses.nct_id.isin(history_ncts)]
            outcome_temp = outcome_temp[(outcome_temp.p_value_modifier.isnull()) | (outcome_temp.p_value_modifier != '>')]
            outcome_temp = outcome_temp[(outcome_temp.p_value >= 0) & (outcome_temp.p_value <= 1) & (outcome_temp.outcome_type == 'Primary')]
            nct2p = outcome_temp.groupby('nct_id').p_value.min()
            if len(nct2p) == 0:
                avg_history_success_rate = 0
            else:
                avg_history_success_rate = sum(nct2p<= 0.05)/len(nct2p)
        
            event_temp = reported_event_totals[reported_event_totals.nct_id.isin(history_ncts)]
            event_temp = event_temp[(event_temp.event_type == 'serious') | (event_temp.event_type == 'deaths')]
            event_temp = event_temp[~event_temp.subjects_affected.isnull()]
            nct2event = event_temp.groupby('nct_id').subjects_affected.sum()
        
            if len(nct2event) == 0:
                avg_history_adverse_events = 0
            else:
                avg_history_adverse_events = sum(nct2event)/len(nct2event)
            
            max_historical_trial_enrollment = studies_temp.enrollment.max()
            avg_historical_trial_enrollment = studies_temp.enrollment.mean()
        
            operation_years = (studies_temp.start_date.max().year - studies_temp.start_date.min().year)
            years_since_last_trial_facility = timestamp.year - studies_temp.start_date.max().year
        
            nct2num_facility = dict(facilities_studies[(facilities_studies.nct_id.isin(history_ncts))].groupby('nct_id').facility_id.agg(len))
            if len(nct2num_facility) == 0:
                in_multi_center_trial = 0
                num_multi_center_trial = 0
            else:
                if min(nct2num_facility.values()) > 1:
                    in_multi_center_trial = 1
                else:
                    in_multi_center_trial = 0
                num_multi_center_trial = sum([1 if j > 1 else 0 for i,j in nct2num_facility.items()])
        
            return [num_trials_conducted, num_conditions_conducted, num_interventions_conducted, num_sponsors_conducted, 
                    num_phase1, num_phase2, num_phase3, num_phase4,
                    avg_history_success_rate,  avg_history_adverse_events, max_historical_trial_enrollment, avg_historical_trial_enrollment,
                    operation_years, years_since_last_trial_facility, in_multi_center_trial, num_multi_center_trial]
        except Exception as e:
            print(e)
    
    train_df['derived_features'] = train_df.parallel_apply(lambda x: get_derived_features(x.facility_id, x.timestamp), axis = 1)

    facility_to_merge_col = []
    for country in facilities.country.value_counts()[:10].keys():
        facilities['in_' + country.lower()] = facilities['country'].apply(lambda x: 1 if x == country else 0)
        facility_to_merge_col.append('in_' + country.lower())

    for state in facilities.state.value_counts()[:10].keys():
        facilities['in_' + state.lower() + '_state'] = facilities['state'].apply(lambda x: 1 if x == state else 0)
        facility_to_merge_col.append('in_' + state.lower()+ '_state')

    for city in facilities.city.value_counts()[:10].keys():
        facilities['in_' + city.lower()+ '_city'] = facilities['city'].apply(lambda x: 1 if x == city else 0)
        facility_to_merge_col.append('in_' + city.lower()+ '_city')

    train_df = train_df.merge(facilities[['facility_id'] + facility_to_merge_col], how = 'left')
    feature_list = "num_trials_conducted, num_conditions_conducted, num_interventions_conducted, num_sponsors_conducted, num_phase1, num_phase2, num_phase3, num_phase4,avg_history_success_rate,  avg_history_adverse_events, max_historical_trial_enrollment, avg_historical_trial_enrollment,operation_years, years_since_last_trial_facility, in_multi_center_trial, num_multi_center_trial".split(',')
    feature_list = [i.strip() for i in feature_list]

    expanded_df = pd.DataFrame(train_df['derived_features'].tolist(), columns=feature_list)
    train_df = train_df.drop(columns=['derived_features'])
    train_df = pd.concat([train_df, expanded_df], axis=1)

    return train_df

In [ ]:
dataset = get_dataset(name='rel-trial', process=False)
task = get_task('rel-trial', 'site-success', download=True)

In [ ]:
val_df = task.get_table("val").df
val_df = get_features(val_df)

In [ ]:
train_df = task.get_table("train").df
train_df = get_features(train_df)

In [ ]:
train_df = train_df.fillna(0)
val_df = val_df.fillna(0)

In [ ]:
from torch_frame import TaskType, stype
from torch_frame.gbdt import LightGBM, XGBoost
from torch_frame.data import Dataset
from torch_frame.typing import Metric
from torch_frame.utils import infer_df_stype

col_to_stype = infer_df_stype(train_df)
col_to_stype

DROP_COLS = [
    # drop identifier cols
    'facility_id',
    'timestamp',
]
for c in DROP_COLS:
    del col_to_stype[c]

train_dset = Dataset(
    train_df.drop(DROP_COLS, axis=1),
    col_to_stype=col_to_stype,
    target_col='success_rate'
).materialize()

val_tf = train_dset.convert_to_tensor_frame(val_df.drop(DROP_COLS, axis=1))
tune_metric = Metric.MAE
print(train_dset.tensor_frame.num_cols, train_dset.tensor_frame.num_rows)

gbdt = LightGBM(TaskType.REGRESSION, metric=tune_metric)
gbdt.tune(tf_train=train_dset.tensor_frame, tf_val=val_tf, num_trials=10)

#gbdt.save('site_lightgbm.json')

In [ ]:
import shap

shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

In [ ]:
from sklearn.metrics import mean_absolute_error, accuracy_score
import numpy as np
pred = gbdt.predict(tf_test=val_tf)
pred_ = np.rint(pred.numpy())  # round to nearest int
# Entire val set
y_ = val_tf.y.numpy()
mae = mean_absolute_error(y_, pred_)
tmae = mean_absolute_error(y_, np.zeros_like(y_))
binacc = accuracy_score((y_ > 0).astype(int), (pred_ > 0).astype(int))

print('ALL VAL')
print(f'Trivial MAE: {tmae:.3f}')
print(f"MAE: {mae:.3f}")
print(f"Binary Accuracy: {binacc:.1%}")
print()

# nonzero entries
nzy = y_[y_ != 0]
nzp = pred_[y_ != 0]
mae = mean_absolute_error(nzy, nzp)
tmae = mean_absolute_error(nzy, np.zeros_like(nzy))

print('NONZERO VAL')
print(f'Trivial MAE: {tmae:.3f}')
print(f"MAE: {mae:.3f}")

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[:5000])
shap_values = explainer.shap_values(val_arr, pred_[:5000])

In [ ]:
# TODO verify
feat_names = val_tf.col_names_dict[stype.categorical] + val_tf.col_names_dict[stype.numerical]

In [ ]:
shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=50,
                  feature_names=feat_names)

In [ ]:
test_df = task.get_table("test").df
test_df = get_features(test_df)
test_tf = train_dset.convert_to_tensor_frame(test_df.drop(DROP_COLS, axis=1))
pred = gbdt.predict(tf_test=test_tf)
task.evaluate(pred)

In [ ]:
task.evaluate(pred_map)

In [ ]:
test_df

In [ ]:
task_name = 'rel-trial-site'
train_df.to_csv(task_name + '-train.csv', index = False)
val_df.to_csv(task_name + '-val.csv', index = False)
test_df.to_csv(task_name + '-test.csv', index = False)